# S&P 500 Derin Ogrenme ile Zaman Serisi Siniflandirma — v3 (Overfitting Duzeltmesi)

Bu notebook, S&P 500 sektor ETF'lerinin makro olaylar etrafindaki fiyat hareketlerini analiz ederek,
**T+5 gun sonra fiyatin yukselip yukselmeyecegini** tahmin eden bir **Ikili Siniflandirma (Binary Classification)** modeli kurar.

---
## v2 -> v3 Gecisinin Sebebi

**v1 durumu:** Underfitting (val_acc > train_acc, loss platoda sikismis ~0.686)

**v2 durumu:** Overfitting — Train loss dusuyor ama Val loss 2. epoch'tan sonra artmaya basladi. `restore_best_weights=True` sayesinde erken duruldu ama test acc %55.75'e geriledi.

**v3 stratejisi:** "Sweet spot" — kapasiteyi biraz azalt, regularizasyonu guclendir, label smoothing ekle.

| Parametre | v2 | v3 |
|---|---|---|
| Dense katmanlar | 128 -> 64 -> 32 | **64 -> 32** (daha kucuk) |
| Dropout orani | 0.2 | **0.35** |
| Weight decay | 0.0001 | **0.001** |
| Learning rate | 0.001 | **0.0005** |
| Label smoothing | yok | **0.05** |
| EarlyStopping patience | 15 | **8** |
| ReduceLROnPlateau patience | 5 | **3** |
| Epoch | 200 | **100** |
| Batch size | 128 | 128 (ayni) |
| Aktivasyon | swish | swish (ayni) |
| BatchNormalization | var | **var** (her iki katmanda) |

---
**Veri Seti:** `sp500_deep_learning_massive_data.csv` (~257.000 satir)
**Model:** Ileri Beslemeli Yapay Sinir Agi (Feedforward / Dense) + BatchNorm + Label Smoothing
**Framework:** TensorFlow / Keras

---
## Kutuphane Yuklemesi

In [ ]:
# ============================================================
# Temel Kutuphaneler
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Sklearn - On Isleme
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras - Derin Ogrenme
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, RMSprop, SGD, AdamW
from tensorflow.keras.losses import BinaryCrossentropy   # YENI v3: Label smoothing icin

# Tekrarlanabilirlik icin rastgelelik tohumunu sabitle
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Surumu: {tf.__version__}")
print(f"NumPy Surumu: {np.__version__}")
print(f"Pandas Surumu: {pd.__version__}")
print("\nTum kutuphaneler basariyla yuklendi!")
print()

---
## Adim 1: Veriyi Okuma ve Hedef Degisken (Y) Olusturma

**Mantik:**
Her `[Olay_Ismi, Hisse]` grubu icinde, kapanis fiyatini 5 gun ileri kaydirarak (`shift(-5)`) T+5 gun sonraki fiyati buluyoruz.
- T+5 fiyati > Bugunku fiyat -> **Hedef = 1 (Yukselis)**
- T+5 fiyati <= Bugunku fiyat -> **Hedef = 0 (Dusus)**

In [ ]:
# ============================================================
# 1.1 - CSV Dosyasini Oku
# ============================================================
DOSYA_YOLU = "sp500_deep_learning_massive_data.csv"

df = pd.read_csv(DOSYA_YOLU)

print(f"Veri Seti Boyutu: {df.shape[0]:,} satir x {df.shape[1]} sutun")
print(f"\nSutunlar:\n{list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# 1.2 - Hedef Degisken (Y) Olusturma
# ============================================================
df = df.sort_values(by=['Olay_Ismi', 'Hisse', 'T0_Goreceli_Gun']).reset_index(drop=True)

df['Fiyat_T5'] = df.groupby(['Olay_Ismi', 'Hisse'])['Duzeltilmis_Kapanis'].shift(-5)

# Hedef degiskeni olustur: 1 = Yukselis, 0 = Dusus
df['Hedef'] = (df['Fiyat_T5'] > df['Duzeltilmis_Kapanis']).astype(int)

# T+5 hesaplanamayan (NaN) satirlari sil
satirlar_once = len(df)
df = df.dropna(subset=['Fiyat_T5']).reset_index(drop=True)
satirlar_sonra = len(df)

print(f"Silinen NaN satir sayisi: {satirlar_once - satirlar_sonra:,}")
print(f"Kalan satir sayisi: {satirlar_sonra:,}")
print(f"\nHedef Degisken Dagilimi:")
print(df['Hedef'].value_counts())
print(f"\nYukselis Orani: %{df['Hedef'].mean()*100:.1f}")

---
## Adim 2: Ozellik Secimi ve 2D Veri Hazirligi

In [ ]:
# ============================================================
# 2.1 - Ozellik Sutunlarini Sec
# ============================================================
OZELLIK_SUTUNLARI = [
    'Log_Getiri',
    'Volatilite_10g',
    'Volatilite_30g',
    'RSI_14',
    'MACD_12_26_9',
    'MACDh_12_26_9',
    'MACDs_12_26_9',
    'BBL_20_2.0',
    'BBU_20_2.0'
]

print(f"Kullanilacak Ozellik Sayisi: {len(OZELLIK_SUTUNLARI)}")
print(f"Ozellikler: {OZELLIK_SUTUNLARI}")

# Eksik deger kontrolu
eksik = df[OZELLIK_SUTUNLARI].isnull().sum()
print(f"\nOzelliklerdeki Eksik Degerler:\n{eksik[eksik > 0]}")

# Eksik degerleri olan satirlari temizle
df = df.dropna(subset=OZELLIK_SUTUNLARI).reset_index(drop=True)
print(f"\nTemizleme sonrasi kalan satir: {len(df):,}")

In [ ]:
# ============================================================
# 2.2 - 2D Veri Hazirligi (Orneklem, Ozellik)
# ============================================================
X = df[OZELLIK_SUTUNLARI].values
y = df['Hedef'].values

ozellik_sayisi = X.shape[1]

print(f"X boyutu: {X.shape}  -> (Orneklem, Ozellik)")
print(f"y boyutu: {y.shape}  -> (Orneklem,)")
print(f"\nToplam orneklem sayisi: {X.shape[0]:,}")
print(f"Ozellik sayisi: {ozellik_sayisi}")

---
## Adim 3: Train/Test Ayrimi ve Olceklendirme (Leakage-Safe)

In [ ]:
# ============================================================
# 3.1 - Train / Test Bolmesi (shuffle=False) + Leakage-safe Scaling
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)

print(f"Bolme oncesi X boyutu: {X.shape}")
print(f"\nEgitim Seti:  X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test Seti:    X_test={X_test.shape},  y_test={y_test.shape}")

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nEgitim Setindeki Yukselis Orani: %{y_train.mean()*100:.1f}")
print(f"Test Setindeki Yukselis Orani:   %{y_test.mean()*100:.1f}")
print(f"\nScaler fit edildi (sadece train): mean={scaler.mean_.round(3)}")

---
## Adim 4: Feedforward (Dense) Model Mimarisi — v3 (Daha Kucuk + Guclu Regularizasyon)

**Yeni Mimari:**
```
Input (9,)
  -> Dense (64, swish) -> BatchNorm -> Dropout (0.35)
  -> Dense (32, swish) -> BatchNorm -> Dropout (0.35)
  -> Dense (1, sigmoid)
```

**v2'den Farklari ve Sebepleri:**
- **Dense katmani sayisi azaltildi (3 -> 2):** v2'de 12,417 parametre 9 ozellik icin fazlaydi. Model ezberliyordu. Daha kucuk model bu veride daha iyi genelleme yapar.
- **Dropout 0.2 -> 0.35:** Overfitting'i agresif sekilde frenlemek icin. 0.5 cok fazlaydi (underfitting), 0.2 cok azdi (overfitting). 0.35 dengeli nokta.
- **BatchNormalization korunuyor:** Gradient stabilitesi icin kritik, problemimiz bu degildi.
- **swish aktivasyonu korunuyor:** Kucuk sinyalli veride ReLU'dan daha iyi calisiyor.

In [ ]:
# ============================================================
# 4.1 - Modeli Kur (v3 Mimarisi)
# ============================================================
model = Sequential([
    Input(shape=(ozellik_sayisi,)),

    # 1. Gizli Katman: 64 noron, swish aktivasyonu
    Dense(64, activation='swish'),
    BatchNormalization(),
    Dropout(0.35),

    # 2. Gizli Katman: 32 noron, swish aktivasyonu
    Dense(32, activation='swish'),
    BatchNormalization(),
    Dropout(0.35),

    # Cikti Katmani: 1 noron, Sigmoid aktivasyonu
    Dense(1, activation='sigmoid')
])

# Model ozetini goster
model.summary()

---
## Adim 5: Modeli Derleme ve Egitme (Compile & Fit) — v3

**Derleme Ayarlari:**
- `loss=BinaryCrossentropy(label_smoothing=0.05)`: **YENI**. Model "%99 eminim" yerine "%94 eminim" demeye zorlanir. Asiri guven ile olusan Val Loss sismesini dogrudan engeller. Finansal veride cok etkili.
- `optimizer=AdamW(lr=0.0005, weight_decay=0.001)`: Learning rate yariya dusuruldu (daha yumusak ogrenme), weight decay 10 kat artirildi (agirlik buyumesini cezalandirma).

**Egitim Ayarlari:**
- `batch_size=128`: v2 ile ayni
- `epochs=100`: v2'de zaten erken duruyordu, 200 gereksizdi
- `EarlyStopping(patience=8)`: v2'de val_loss 2. epoch'ta minimumdu. Patience 15 gereksiz uzundu; 8 daha hizli tepki verir.
- `ReduceLROnPlateau(patience=3)`: Daha hizli LR azaltma ile plato erken kirilir.

In [ ]:
# ============================================================
# 5.1 - Modeli Derle (Compile) - v3
# ============================================================
# YENI v3: Label smoothing ile loss fonksiyonu
loss_fn = BinaryCrossentropy(label_smoothing=0.05)

# YENI v3: LR 0.001 -> 0.0005, weight_decay 0.0001 -> 0.001
ozel_optimizer = AdamW(learning_rate=0.0005, weight_decay=0.001)

model.compile(
    loss=loss_fn,
    optimizer=ozel_optimizer,
    metrics=['accuracy']
)

print("Model derlendi.")
print(f"   Loss: BinaryCrossentropy(label_smoothing=0.05)")
print(f"   Optimizer: AdamW(learning_rate=0.0005, weight_decay=0.001)")
print(f"   Metrik: Accuracy")

In [ ]:
# ============================================================
# 5.2 - Callback'leri Tanimla - v3
# ============================================================
# YENI v3: patience 15 -> 8
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,    # KRITIK: En iyi epoch'un agirliklarini geri yukler
    verbose=1
)

# YENI v3: patience 5 -> 3 (daha hizli tepki)
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("Callback'ler hazirlandi:")
print("   EarlyStopping(patience=8, restore_best_weights=True)")
print("   ReduceLROnPlateau(factor=0.5, patience=3)")

In [ ]:
# ============================================================
# 5.3 - Modeli Egit (Fit) - v3
# ============================================================
print("Model egitimi basliyor...\n")

gecmis = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,                        # v3: 200 -> 100
    batch_size=128,
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

print("\nEgitim tamamlandi!")

---
## Adim 6: Model Performansini Gorsellestir ve Degerlendir

In [ ]:
# ============================================================
# 6.1 - Egitim Surecini Gorsellestir (Loss & Accuracy Grafikleri)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Sol Grafik: Loss (Kayip) ---
axes[0].plot(gecmis.history['loss'], label='Egitim Kaybi (Train Loss)', linewidth=2)
axes[0].plot(gecmis.history['val_loss'], label='Dogrulama Kaybi (Val Loss)', linewidth=2, linestyle='--')
axes[0].set_title('Kayip Fonksiyonu (Loss) - Epoch Bazinda', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss (Label Smoothed)')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Sag Grafik: Accuracy (Dogruluk) ---
axes[1].plot(gecmis.history['accuracy'], label='Egitim Dogrulugu (Train Acc)', linewidth=2)
axes[1].plot(gecmis.history['val_accuracy'], label='Dogrulama Dogrulugu (Val Acc)', linewidth=2, linestyle='--')
axes[1].set_title('Dogruluk (Accuracy) - Epoch Bazinda', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('egitim_grafikleri_v3.png', dpi=150, bbox_inches='tight')
plt.show()

# Overfitting tespiti icin bilgi notu
son_train_loss = gecmis.history['loss'][-1]
son_val_loss = gecmis.history['val_loss'][-1]
fark = abs(son_train_loss - son_val_loss)
print(f"\nSon Epoch Istatistikleri:")
print(f"   Train Loss: {son_train_loss:.4f}")
print(f"   Val Loss:   {son_val_loss:.4f}")
print(f"   Fark:       {fark:.4f}")
if fark > 0.1:
    print("   Egitim ve dogrulama kaybi arasinda belirgin fark var -> Overfitting riski!")
else:
    print("   Egitim ve dogrulama kaybi yakin -> Model dengeli ogrenmis.")

# En iyi epoch bilgisi
en_iyi_epoch = int(np.argmin(gecmis.history['val_loss'])) + 1
en_iyi_val_loss = min(gecmis.history['val_loss'])
print(f"\nEn iyi epoch: {en_iyi_epoch} (Val Loss: {en_iyi_val_loss:.4f})")
print("(restore_best_weights=True sayesinde modelde bu epoch'un agirliklari var)")

In [ ]:
# ============================================================
# 6.2 - Test Seti Uzerinde Nihai Degerlendirme
# ============================================================

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("=" * 60)
print("MODEL DEGERLENDIRME OZETI (v3)")
print("=" * 60)
print(f"\n   Test Kaybi (Loss):     {test_loss:.4f}")
print(f"   Test Dogrulugu (Acc):  %{test_acc*100:.2f}")
print(f"\n   Egitim Epoch Sayisi:   {len(gecmis.history['loss'])}")
print(f"   Ozellik Sayisi:        {ozellik_sayisi}")
print(f"   Toplam Parametre:      {model.count_params():,}")
print(f"   Mimari:                Dense(64)->BN->Drop(0.35) -> Dense(32)->BN->Drop(0.35) -> Dense(1)")
print(f"   Aktivasyonlar:         swish (gizli) / Sigmoid (cikti)")
print(f"   Regularizasyon:        Dropout(0.35) + BatchNorm + LabelSmoothing(0.05) + EarlyStopping(patience=8, restore_best=True) + ReduceLROnPlateau(patience=3)")
print(f"   Optimizer:             AdamW(lr=0.0005, weight_decay=0.001)")
print(f"   Loss:                  BinaryCrossentropy(label_smoothing=0.05)")
print(f"   Batch Size:            128")
print("\n" + "=" * 60)
print("Analiz tamamlandi!")
print("=" * 60)

---
## Adim 7: v1 - v2 - v3 Karsilastirma ve Yorumlama

### Referans Sonuclar
| Versiyon | Test Loss | Test Acc | Sorun / Yorum |
|---|---|---|---|
| v1 (orijinal) | 0.6867 | %55.87 | Underfitting — Val acc > Train acc |
| v2 (daha buyuk) | 0.6885 | %55.75 | Overfitting — Val loss 2. epoch'tan sonra patladi |
| **v3 (hedef)** | **0.675 — 0.680** | **%57 — %58** | Sweet spot — dengeli ogrenme |

### v3 Loss grafiginde ne aramalisin?

**Iyi senaryo:**
- Train ve Val loss egrilerinin **birlikte duserek** ilerlemesi
- Val loss'un minimumu **epoch 5-15 arasinda** olmasi (v2'de 2. epoch'tan patliyordu)
- Aralarindaki fark 0.01'den az

**Hala sorun varsa:**
- Val loss hala erken patliyorsa -> Dropout 0.35 -> 0.45 yapmayi dene
- Model hic ogrenmiyorsa (her iki loss da duz) -> Label smoothing 0.05 -> 0.02 azalt, LR 0.001'e geri cikar

### Label smoothing hakkinda not

Label smoothing aktif oldugu icin **loss degerleri v1/v2 ile dogrudan karsilastirilamaz** — teorik minimum artik 0 degil. Bu yuzden asil onemli metrik **Test Accuracy** olacak.

### Eger v3 de %56-57'de sikisiyorsa

Bu noktada hiperparametre ile alabilecegin yerin sonuna geldin demektir. Model problemi degil, **veri problemi** var. Su adimlar denenebilir:

1. **Ozellik muhendisligi:** RSI esik bayraklari (RSI > 70, RSI < 30), MACDh'nin 3-gunluk turevi, BB bant genisligi, fiyatin BB'ye gore konumu
2. **Hedef tanimini netlestir:** T+5 yerine "T+5 fiyati bugune gore en az %1 yukselmis mi?" sorusu (gurultulu notr durumlari filtreler)
3. **Sektor-goreli ozellikler:** Hissenin kendi sektoruyle gorece performansi
4. **Olay tipine gore segmentasyon:** TUFE olaylari icin ayri model, faiz kararlari icin ayri model

Bu adimlar "apayri sistem" degildir, mevcut pipeline'a ozellik eklemektir.